# Radón en Minnesota: tres modelos y un solo cambio de línea

Este notebook es el acompañamiento del vídeo sobre modelos jerárquicos. No es el material
del seminario: es el sitio donde lo que allí se cuenta de palabra se ejecuta, se rompe y
se mira por dentro.

Los datos son 919 mediciones de radón en viviendas de 85 condados de Minnesota, el
conjunto que usan Gelman y Hill y que lleva veinte años siendo el ejemplo canónico de
modelo jerárquico. El radón es un gas radiactivo que se cuela desde el suelo; la EPA
recomienda actuar por encima de 4 pCi/L. La pregunta es de las que tienen consecuencias:
**¿cuánto radón cabe esperar en una vivienda concreta de un condado concreto?**

El recorrido:

1. Los datos, y el problema que traen de fábrica.
2. **Modelo 1**: todos los condados juntos. La regresión de siempre, en OLS y en bayesiano.
3. **Modelo 2**: un nivel propio por condado, cada uno estimado solo.
4. **Modelo 3**: el jerárquico. Cambia una línea.
5. *Shrinkage*: por qué los condados con pocos datos se acercan a la media, y por qué eso
   no es hacer trampa.
6. Diagnóstico: si el muestreador no ha funcionado, la posteriori no opina.
7. Para qué sirve: la predicción para una casa.

No hay ejercicios. Se ejecuta de arriba abajo y se lee.

---

**Cómo ejecutar esto.** En Google Colab no hay que instalar ni descargar nada: la primera
celda de código se ocupa de las dependencias y los datos se leen por URL. Si Colab pide
reiniciar el entorno después de instalar, se reinicia y se vuelve a empezar por arriba. En
local, un entorno con `numpy`, `pandas`, `matplotlib`, `statsmodels`, `pymc` y `arviz`.

Los tres modelos juntos tardan un par de minutos.

In [ ]:
# En Colab: instala lo que falta. En local no hace nada: se da por hecho el entorno.
try:
    import google.colab  # noqa: F401

    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    %pip install -q "numpy>=1.26,<2.5" "pymc>=5.15,<6" "arviz>=0.23,<1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

import pymc as pm
import arviz as az
import statsmodels.api as sm

ACENTO = "#800080"
GRISES = ["#000000", "#4A4A4A", "#7A7A7A", "#AAAAAA"]

plt.rcParams.update({
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#E6D6E6",
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler("color", GRISES),
    "axes.titlecolor": ACENTO,
    "figure.dpi": 110,
    "font.size": 13,
})

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. 919 mediciones en 85 condados

Cada fila es una vivienda medida. De todas las columnas del fichero original solo hacen
falta cuatro:

- `log_radon`: el logaritmo de la medición en pCi/L. En logaritmos porque el radón se
  reparte de forma muy asimétrica, y porque así los efectos se leen como porcentajes.
- `floor`: 0 si la medición se hizo en el sótano, 1 si en la planta baja.
- `county` y `county_code`: el condado, con nombre y con índice de 0 a 84.

In [ ]:
# El fichero de Gelman, leído del repositorio de ejemplos de PyMC. Fijado al tag
# 2026.02.0: en `main` alguien puede mover el fichero cualquier martes por la tarde.
URL_DATOS = (
    "https://raw.githubusercontent.com/pymc-devs/pymc-examples/"
    "2026.02.0/examples/data/radon.csv"
)

df = pd.read_csv(URL_DATOS)[["county", "county_code", "floor", "log_radon"]]

floor = df["floor"].values
log_radon = df["log_radon"].values
county_idx = df["county_code"].values
n_condados = df["county_code"].nunique()

print(f"{len(df)} mediciones en {n_condados} condados")
df.head()

### El sótano da niveles más altos que la planta baja

Que es justo lo que uno esperaría de un gas que sube desde el suelo. Esta es la parte
fácil del problema y la que cualquier regresión resuelve sola.

In [ ]:
# Las dos distribuciones enteras, apiladas: con 766 mediciones en sótano y 153 en planta
# baja, cualquier recuento sin normalizar aplastaría al segundo grupo.
grupos = [
    ("Planta baja", df.loc[df["floor"] == 1, "log_radon"].values),
    ("Sótano", df.loc[df["floor"] == 0, "log_radon"].values),
]

rejilla = np.linspace(-3, 4.5, 400)

fig, ax = plt.subplots(figsize=(8, 4))
for i, (nombre, valores) in enumerate(grupos):
    densidad = gaussian_kde(valores)(rejilla)
    densidad = densidad / densidad.max() * 0.85
    ax.fill_between(rejilla, i, i + densidad, color=GRISES[2 - i], alpha=0.75, zorder=2 - i)
    ax.plot(rejilla, i + densidad, color="black", linewidth=1, zorder=2 - i)

    mediana = np.median(valores)
    ax.vlines(mediana, i, i + np.interp(mediana, rejilla, densidad), color=ACENTO,
              linewidth=2.5, zorder=3)

ax.set_yticks([i + 0.1 for i in range(len(grupos))])
ax.set_yticklabels([nombre for nombre, _ in grupos])
ax.set_xlabel("log(radón)")
ax.set_ylim(-0.1, 2.1)
ax.grid(axis="y", visible=False)
plt.show()

### Pero hay condados con cien mediciones y condados con una

Y esta es la parte difícil. Los 919 datos no están repartidos: ST LOUIS tiene 116
mediciones y hay tres condados con una sola.

In [ ]:
conteo = df["county"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(conteo.values, bins=30, edgecolor="white", color="#666666", alpha=0.85)
ax.set_xlabel("Mediciones por condado")
ax.set_ylabel("Número de condados")
ax.set_title(
    f"Mediana: {int(conteo.median())} mediciones · "
    f"{(conteo <= 3).sum()} condados con 3 o menos"
)
plt.show()

print(conteo.head(3).to_string())
print("...")
print(conteo.tail(3).to_string())

Ese desequilibrio condiciona todo lo que viene después, y es el motivo de que haya tres
modelos y no uno. Con 116 mediciones un condado se defiende solo. Con una, no.

Las dos salidas obvias son malas:

- **Juntarlo todo** y estimar un único nivel para Minnesota. Barato, estable y falso: los
  condados no son iguales.
- **Separarlo todo** y estimar cada condado por su cuenta. Honesto sobre las diferencias
  y ruinoso donde hay una medición.

El jerárquico es la tercera, y no es un punto medio arbitrario: es dejar que los datos
decidan cuánto se parecen los condados entre sí.

## 2. Modelo 1: todos los condados juntos

Empezamos por lo conocido. Una recta, sin condados:

$$\log(\text{radón}_i) = \beta_0 + \beta_1 \cdot \text{planta}_i + \varepsilon_i$$

En OLS esto son dos líneas y lo hemos hecho todos mil veces.

In [ ]:
X = sm.add_constant(df[["floor"]])
ols = sm.OLS(df["log_radon"], X).fit()

pd.DataFrame({
    "Coeficiente": ols.params.round(3),
    "Error estándar": ols.bse.round(3),
    "IC 2,5%": ols.conf_int()[0].round(3),
    "IC 97,5%": ols.conf_int()[1].round(3),
    "p-valor": ols.pvalues.round(3),
})

### El mismo modelo, en bayesiano

$$y_i \sim \mathcal{N}(\alpha + \beta_1 x_i, \; \sigma^2)$$

Lo que en OLS estaba implícito, aquí hay que escribirlo: qué distribución tiene el error y
qué creemos de los parámetros antes de mirar los datos.

**¿De dónde salen los números de las prioris?** De la escala de la variable, no del deseo
de que sean amplias:

- `log_radon` tiene desviación típica **0,82** y va de −2,3 a 3,9.
- $\alpha \sim \mathcal{N}(0, 2^2)$ cubre de 0,02 a 55 pCi/L. De sobra para un nivel base.
- $\beta_1 \sim \mathcal{N}(0, 1^2)$ no impone dirección y admite hasta un factor 7 entre
  plantas.
- $\sigma \sim \text{Exp}(1)$: positiva, con masa donde está la escala de los datos.

Una $\mathcal{N}(0, 10^2)$ para $\alpha$, la priori "no informativa" de manual, son doce
desviaciones típicas de la variable. Eso no es prudencia; es no haber mirado.

In [ ]:
with pm.Model() as modelo_agrupado:
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    beta1 = pm.Normal("beta1", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    y_obs = pm.Normal("y_obs", mu=alpha + beta1 * floor, sigma=sigma, observed=log_radon)

### Y esto no se afirma: se simula antes de mirar los datos

Una priori no se defiende con adjetivos. Se generan datos desde ella, se miran en las
unidades del problema —aquí, pCi/L— y se comprueba si son creíbles. Es la parte del
trabajo que casi nadie hace y la que más disgustos ahorra.

`simular_previa` monta el mismo modelo con las escalas que se le pasen y devuelve las
mediciones de radón que implica, sin haber visto un solo dato.

In [ ]:
def simular_previa(escala_alpha, escala_beta):
    """Devuelve mediciones de radón (pCi/L) simuladas desde la priori, sin ver los datos."""
    with pm.Model():
        alpha = pm.Normal("alpha", mu=0, sigma=escala_alpha)
        beta1 = pm.Normal("beta1", mu=0, sigma=escala_beta)
        sigma = pm.Exponential("sigma", 1)
        pm.Normal("y_obs", mu=alpha + beta1 * floor, sigma=sigma, shape=len(floor))

        previa = pm.sample_prior_predictive(draws=500, random_seed=SEMILLA)

    return np.exp(previa.prior["y_obs"].values.reshape(-1))


radon_sensato = simular_previa(2, 1)
radon_amplio = simular_previa(10, 10)

for nombre, muestras in [("De la escala", radon_sensato),
                         ("Amplias", radon_amplio)]:
    p50, p90, p99 = np.percentile(muestras, [50, 90, 99])
    print(f"{nombre:>12} → mediana {p50:5.2f} pCi/L | p90 {p90:9.3g} | p99 {p99:9.3g} "
          f"| P(> 1000 pCi/L) = {(muestras > 1000).mean():.1%}")

La priori amplia no es neutral: es una afirmación disparatada sobre el mundo. Dice que una
de cada cuatro viviendas de Minnesota supera los 1000 pCi/L, un nivel que en el mundo real
se ha medido un puñado de veces y que sale en las noticias cuando pasa. Su percentil 90
está en seis cifras. Y el modelo se lo cree, porque un modelo se cree lo que le escribas.

Antes de que alguien lo diga: la priori de la escala tampoco es inocente, y su percentil 99
se va a varios cientos de pCi/L. Es lo que tiene una lognormal con $\sigma \sim \text{Exp}(1)$:
cola gorda. La diferencia entre las dos no es de matiz, es de veinte órdenes de magnitud.

Y con 919 datos todo esto acabará dando igual, que es lo cómodo del ejemplo: los datos
aplastan la priori y las dos versiones llegan al mismo sitio. Con veinte datos, no da igual.
En el Modelo 2 vamos a tener condados con uno.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bordes = np.linspace(-4, 20, 70)
ax.hist(np.log10(radon_amplio), bins=bordes, color="#BBBBBB", label="prioris amplias")
ax.hist(np.log10(radon_sensato), bins=bordes, color=ACENTO, alpha=0.9, label="prioris de la escala")
ax.axvline(np.log10(4), color="black", linestyle="--", linewidth=1)
ax.text(np.log10(4) + 0.3, ax.get_ylim()[1] * 0.9, "límite EPA\n(4 pCi/L)", fontsize=10)
ax.set_xlabel("log₁₀ del radón simulado (pCi/L)")
ax.set_ylabel("Simulaciones")
ax.set_yticks([])
ax.legend()
plt.show()

In [ ]:
with modelo_agrupado:
    idata_agrupado = pm.sample(
        draws=1000, tune=1000, chains=4, random_seed=SEMILLA, progressbar=False
    )

### Con 919 datos, OLS y el bayesiano coinciden

Y conviene decirlo en voz alta, porque hay quien vende lo bayesiano como si cambiara los
números. No los cambia. Con datos de sobra y prioris razonables, la media de la posteriori
y el estimador de máxima verosimilitud se dan la mano.

In [ ]:
resumen = az.summary(idata_agrupado, var_names=["alpha", "beta1", "sigma"])

pd.DataFrame({
    "OLS": [ols.params["const"], ols.params["floor"], np.sqrt(ols.scale)],
    "Bayesiano (media)": resumen["mean"].values,
    "HDI 3%": resumen["hdi_3%"].values,
    "HDI 97%": resumen["hdi_97%"].values,
    "r_hat": resumen["r_hat"].values,
}, index=["alpha / const", "beta1 / floor", "sigma"]).round(3)

La diferencia no está en el punto central. Está en lo que te llevas además de él: una
distribución completa de cada parámetro, con la que se puede calcular la probabilidad de
cualquier cosa que le interese a alguien. Volvemos a ello en la sección 7.

Lo que este modelo no hace es responder a la pregunta con la que empezamos. Dice cuánto
radón hay en Minnesota. No dice cuánto hay en tu condado.

## 3. Modelo 2: un nivel propio por condado

La reacción natural: si los condados son distintos, que cada uno tenga su nivel. Un
$\alpha$ por condado, 85 en total, y la pendiente y $\sigma$ compartidas.

En código es una palabra: `shape=n_condados`.

In [ ]:
with pm.Model() as modelo_unpooled:
    alpha = pm.Normal("alpha", mu=0, sigma=2, shape=n_condados)  # un alpha por condado
    beta1 = pm.Normal("beta1", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    y_obs = pm.Normal(
        "y_obs", mu=alpha[county_idx] + beta1 * floor, sigma=sigma, observed=log_radon
    )

    idata_unpooled = pm.sample(
        draws=1000, tune=1000, chains=4, random_seed=SEMILLA, progressbar=False
    )

### Y con pocas mediciones se dispara la incertidumbre

Los condados van ordenados de menos a más datos. Abajo, los que tienen una o dos
mediciones: barras larguísimas, porque el modelo no tiene con qué. Arriba, los que tienen
decenas.

In [ ]:
alpha_unpooled = idata_unpooled.posterior["alpha"].values.reshape(-1, n_condados)
media_unpooled = alpha_unpooled.mean(axis=0)
hdi_unpooled = az.hdi(alpha_unpooled[np.newaxis, :, :], hdi_prob=0.94)

n_obs = df["county_code"].value_counts().sort_index().values
orden = np.argsort(n_obs)

fig, ax = plt.subplots(figsize=(8, 4.5))
y_pos = np.arange(n_condados)
ax.errorbar(
    media_unpooled[orden], y_pos,
    xerr=[media_unpooled[orden] - hdi_unpooled[orden, 0],
          hdi_unpooled[orden, 1] - media_unpooled[orden]],
    fmt="o", markersize=3, color="black", ecolor="#888888", alpha=0.7, elinewidth=1,
)
ax.set_xlabel("α (nivel base de log(radón))")
ax.set_ylabel("Condados, de menos a más datos")
ax.set_yticks([])
plt.show()

anchura = hdi_unpooled[:, 1] - hdi_unpooled[:, 0]
print(f"Anchura del HDI en el condado con menos datos: {anchura.max():.2f}")
print(f"Anchura del HDI en el condado con más datos:   {anchura.min():.2f}")

Este modelo tampoco sirve. No porque mienta —es el más honesto de los tres sobre lo que
no sabe—, sino porque tira a la basura información que tiene delante. Un condado con una
medición no sabe nada de sí mismo, pero los otros 84 condados de Minnesota saben bastante
sobre qué niveles de radón son plausibles allí, y este modelo se niega a mirarlos.

Un apunte de letra pequeña: en realidad ni siquiera está del todo suelto, porque la
$\mathcal{N}(0, 2^2)$ de la priori ya está tirando de los condados con una medición hacia
el 0. Es un tirón que hemos puesto nosotros a mano y a ojo. El siguiente modelo hace lo
mismo, pero con un tirón estimado a partir de los datos.

## 4. Modelo 3: los condados se prestan información

La idea: los 85 niveles no son 85 números sueltos, son 85 muestras de una misma
distribución. Minnesota tiene un nivel medio y una variabilidad entre condados, y ambos
se estiman.

$$
\begin{aligned}
y_i &\sim \mathcal{N}(\alpha_{c[i]} + \beta_1 x_i, \; \sigma^2) \\[4pt]
\alpha_c &\sim \mathcal{N}(\mu_\alpha, \sigma_\alpha^2) \\[4pt]
\mu_\alpha &\sim \mathcal{N}(0, 2^2), \qquad \sigma_\alpha \sim \text{Exp}(1)
\end{aligned}
$$

La priori de los $\alpha_c$ ya no la escribimos nosotros: la escriben los datos. Y en
código, comparado con el modelo anterior, cambia **una línea**.

In [ ]:
with pm.Model() as modelo_jerarquico:
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=2)
    sigma_alpha = pm.Exponential("sigma_alpha", 1)

    alpha = pm.Normal("alpha", mu=mu_alpha, sigma=sigma_alpha, shape=n_condados)  # <—
    beta1 = pm.Normal("beta1", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    y_obs = pm.Normal(
        "y_obs", mu=alpha[county_idx] + beta1 * floor, sigma=sigma, observed=log_radon
    )

    idata_jerarquico = pm.sample(
        draws=1000, tune=1000, chains=4, target_accept=0.95,
        random_seed=SEMILLA, progressbar=False,
    )

Los tres modelos son el mismo con distinta restricción sobre los $\alpha_c$:

- **Agrupado**: restricción dura. Todos los $\alpha_c$ valen lo mismo. Equivale a
  $\sigma_\alpha = 0$.
- **Unpooled**: sin restricción entre condados. Equivale a $\sigma_\alpha = \infty$.
- **Jerárquico**: restricción blanda. $\sigma_\alpha$ regula la fuerza, y **se estima**.

Los dos primeros son casos extremos del tercero, con el parámetro clavado a mano en un
extremo o en el otro. Por eso no hay que elegir entre "juntar" y "separar": hay que dejar
que el número salga de los datos.

> **Nota.** Quien haya leído sobre jerárquicos habrá visto la *parametrización no
> centrada*, el truco de escribir $\alpha_c = \mu_\alpha + \sigma_\alpha \cdot z_c$ para
> que el muestreador no se atasque en el embudo que se forma cuando $\sigma_\alpha$ se
> acerca a cero. Aquí no hace falta: con 919 mediciones, 85 condados y una
> $\sigma_\alpha$ que la posteriori mantiene claramente por encima de cero, el modelo
> centrado muestrea limpio con `target_accept=0.95`. El truco se saca cuando aparecen
> divergencias, y en la sección 6 comprobamos que aquí no aparecen.

## 5. *Shrinkage*: los condados con pocos datos se acercan a la media

Aquí está el efecto entero en un gráfico. Cada línea gris une la estimación *unpooled* de
un condado (punto negro) con la jerárquica (círculo morado). El eje horizontal es cuántas
mediciones tiene.

In [ ]:
alpha_jer = idata_jerarquico.posterior["alpha"].values.reshape(-1, n_condados)
media_jer = alpha_jer.mean(axis=0)
mu_global = float(idata_jerarquico.posterior["mu_alpha"].values.mean())

fig, ax = plt.subplots(figsize=(8, 4.5))
for i in range(n_condados):
    ax.plot([n_obs[i], n_obs[i]], [media_unpooled[i], media_jer[i]],
            color="#BBBBBB", linewidth=0.8, zorder=0)
ax.scatter(n_obs, media_unpooled, color="black", s=25, alpha=0.65, label="Unpooled")
ax.scatter(n_obs, media_jer, facecolors="white", edgecolors=ACENTO, s=30,
           linewidths=1.3, label="Jerárquico")
ax.axhline(mu_global, color="black", linestyle="--", linewidth=1, label="Media global")
ax.set_xlabel("Mediciones en el condado")
ax.set_ylabel("α estimado")
ax.legend()
plt.show()

A la izquierda, líneas largas: los condados con dos o tres mediciones se van casi enteros
hacia la media de Minnesota. A la derecha, líneas que no existen: con cuarenta mediciones
el modelo no tira de nadie, porque no le hace falta.

Eso se llama *shrinkage* y tiene una forma cerrada muy fácil de contar. El modelo estima
dos variabilidades: cuánto se diferencian los condados entre sí ($\sigma_\alpha$) y cuánto
se diferencian las viviendas de un mismo condado ($\sigma$). El cociente de sus cuadrados
dice cuántas mediciones propias hacen falta para que un condado pese tanto como lo que se
sabe de Minnesota.

In [ ]:
s_alpha = float(idata_jerarquico.posterior["sigma_alpha"].values.mean())
s_resid = float(idata_jerarquico.posterior["sigma"].values.mean())
n_equivalente = (s_resid / s_alpha) ** 2

print(f"σ_alpha (entre condados)   = {s_alpha:.2f}")
print(f"σ       (entre viviendas)  = {s_resid:.2f}")
print(f"Lo que el modelo sabe de Minnesota pesa como {n_equivalente:.0f} mediciones propias.")

Por debajo de esas mediciones manda Minnesota, por encima manda el condado, y justo ahí el
tirón se reparte a medias. No hay ningún umbral que hayamos elegido: el número sale de los
datos.

Y no, no es hacer trampa. Es exactamente lo que hace cualquiera al estimar un condado que
no conoce: partir de lo que sabe de los alrededores y corregir con lo que vea. La
diferencia es que aquí está escrito, es reproducible y viene con su incertidumbre.

## 6. Antes de leer la posteriori, mira si el muestreador ha funcionado

Un modelo bayesiano no se resuelve con una fórmula: se muestrea. Y el muestreador puede
fallar. Leer una posteriori sin comprobar esto es como leer la salida de un optimizador
sin mirar si convergió.

Lo primero, la traza. Cada cadena arranca en un sitio distinto y todas tienen que acabar
recorriendo lo mismo: el patrón que se busca es una oruga peluda.

In [ ]:
az.plot_trace(
    idata_jerarquico,
    var_names=["mu_alpha", "sigma_alpha", "beta1", "sigma"],
    chain_prop={"color": GRISES, "linestyle": ["-", "--", ":", "-."]},
    figsize=(9, 7),
)
plt.tight_layout()
plt.show()

Y los números que lo confirman:

- **`r_hat`** compara la varianza entre cadenas con la de dentro de cada cadena. Tiene que
  rondar 1; por encima de 1,01 hay que preocuparse.
- **`ess_bulk`** y **`ess_tail`** cuentan cuántas muestras efectivamente independientes
  sostienen el centro y las colas de la distribución. Unos cientos bastan para el centro;
  para percentiles extremos hace falta más.
- **Divergencias**: pasos que NUTS descarta por inestabilidad numérica. En un jerárquico la
  sospechosa habitual es el embudo alrededor de $\sigma_\alpha$ cuando se acerca a cero, y
  es la razón del `target_accept=0.95`.

In [ ]:
resumen_jer = az.summary(
    idata_jerarquico, var_names=["mu_alpha", "sigma_alpha", "beta1", "sigma"]
)
n_divergencias = int(idata_jerarquico.sample_stats["diverging"].sum())

print(f"Divergencias: {n_divergencias}")

pd.DataFrame({
    "media": resumen_jer["mean"],
    "HDI 3%": resumen_jer["hdi_3%"],
    "HDI 97%": resumen_jer["hdi_97%"],
    "r_hat": resumen_jer["r_hat"],
    "ESS (centro)": resumen_jer["ess_bulk"].astype(int),
    "ESS (colas)": resumen_jer["ess_tail"].astype(int),
})

Cero divergencias, `r_hat` en 1 y ESS de sobra. Esto dice que el muestreador ha explorado
bien la posteriori que le pedí. **No dice que la posteriori sea buena**: son dos preguntas
distintas y esta es la fácil.

La otra se ataca comparando los datos que el modelo genera con los que ha visto.

In [ ]:
with modelo_jerarquico:
    pm.sample_posterior_predictive(
        idata_jerarquico, extend_inferencedata=True,
        random_seed=SEMILLA, progressbar=False,
    )

ax = az.plot_ppc(
    idata_jerarquico, num_pp_samples=100, random_seed=SEMILLA,
    colors=["#AAAAAA", "black", ACENTO], figsize=(8, 4),
)
for linea in ax.get_lines():
    linea.set_linestyle("-")
ax.set_xlabel("log(radón)")
ax.set_ylabel("Densidad")
ax.legend(fontsize=10)
plt.show()

Las réplicas cubren los datos y la forma general está bien. Si te fijas, se comen un pico
que los datos tienen un poco por encima de 1: el modelo es normal y los datos, un poco
menos. Ahí hay margen de mejora si el problema lo pidiera.

Y que se parezcan tampoco demuestra que el modelo sea cierto: ha visto esos datos. Es una
prueba fácil de aprobar que conviene hacer igualmente, porque los suspensos son
carísimamente informativos.

## 7. Para qué sirve todo esto

La pregunta con la que empezamos no era sobre parámetros. Era: **compro una casa en este
condado, ¿cuánto radón me voy a encontrar?**

Con la posteriori en la mano hay dos respuestas distintas, y confundirlas es el error caro:

- El **nivel medio del condado**: dónde está el centro de la distribución allí.
- El **nivel de una vivienda concreta**: ese centro más la variación entre viviendas del
  mismo condado, que es $\sigma$ y es grande.

Comparamos dos condados extremos: ST LOUIS, con 116 mediciones, y LAC QUI PARLE, con 2.

In [ ]:
codigos = df.drop_duplicates("county").set_index("county")["county_code"]
post = idata_jerarquico.posterior


def prediccion(condado, planta=0):
    """Nivel medio del condado y medición de una vivienda, en pCi/L."""
    c = int(codigos[condado])
    a_c = post["alpha"].values[:, :, c].reshape(-1)
    b = post["beta1"].values.reshape(-1)
    s = post["sigma"].values.reshape(-1)

    mu = a_c + b * planta

    # 20 viviendas simuladas por cada muestra de la posteriori: con una sola, el
    # intervalo de la vivienda sale demasiado ruidoso para compararlo entre condados.
    y = rng.normal(mu[:, None], s[:, None], size=(len(mu), 20)).reshape(-1)

    # Los intervalos se calculan en logaritmos, que es donde el modelo trabaja y donde
    # las distribuciones son simétricas, y se devuelven a pCi/L con exp. Ojo con el
    # nivel del condado: exp(mu) es su mediana, y su media es exp(mu + sigma²/2).
    hdi_mu = np.exp(az.hdi(mu + s**2 / 2, hdi_prob=0.94))
    hdi_y = np.exp(az.hdi(y, hdi_prob=0.94))
    return {
        "Condado 3%": hdi_mu[0],
        "Condado 97%": hdi_mu[1],
        "Condado ×": hdi_mu[1] / hdi_mu[0],
        "Vivienda 3%": hdi_y[0],
        "Vivienda 97%": hdi_y[1],
        "Vivienda ×": hdi_y[1] / hdi_y[0],
        "P(> 4 pCi/L)": (np.exp(y) > 4).mean(),
    }


condados = ["ST LOUIS", "LAC QUI PARLE"]
pd.DataFrame(
    [prediccion(c) for c in condados],
    index=[f"{c} ({(df['county'] == c).sum()} mediciones)" for c in condados],
).round(3)

Las columnas con el signo × son el cociente entre los dos extremos del intervalo, que es
como hay que comparar anchuras cuando se trabaja en logaritmos. Léelas y sale solo:

- El intervalo **del condado** pasa de un factor pequeño a uno varias veces mayor al bajar
  de 116 mediciones a 2. Es lo esperable: de LAC QUI PARLE sabemos poco.
- El intervalo **de la vivienda** apenas se mueve entre uno y otro, mientras el del condado
  más que se duplica. Lo domina $\sigma$, la variación entre casas del mismo condado, y esa
  no baja por muchos vecinos que midan.

Saber el condado te sitúa el condado, no tu casa. Por eso la recomendación sanitaria es
medir cada vivienda, y por eso un modelo que solo diera la media del condado sería peor
que inútil: daría tranquilidad falsa.

Y aun así el modelo no decide por ti. Te da la distribución del radón esperable; decidir
si ventilas, sellas o no haces nada exige poner precio a cada error. Eso es teoría de la
decisión, y es otra conversación.

## Lo que te llevas

- Agrupar y separar son el mismo modelo con un parámetro clavado a mano en un extremo. El
  jerárquico lo estima.
- La priori se elige siempre, incluso cuando se cree que no se está eligiendo. Simula desde
  ella antes de ajustar, no después.
- Diagnostica antes de leer la posteriori. Un muestreador que no ha convergido no opina.
- La respuesta es una distribución, no un punto y un asterisco.

**Para seguir.** Gelman y Hill, *Data Analysis Using Regression and Multilevel/Hierarchical
Models*, de donde salen estos datos; y McElreath, *Statistical Rethinking*, con sus clases
en YouTube.

---

Esto es el regalo de suscripción de mi newsletter, donde escribo sobre estadística y
análisis de datos con más sarcasmo del recomendable: **[leonardohansa.com](https://leonardohansa.com)**.

Las slides del seminario del que sale este notebook están publicadas en
**[lhansa.github.io/seminar-bayes](https://lhansa.github.io/seminar-bayes/)**.

Si ejecutas esto, lo rompes o lo mejoras, cuéntamelo respondiendo a cualquier correo. Los
leo todos.